1. What is Generative AI and what are its primary use cases across industries?

Generative AI is a type of Artificial Intelligence that can create new content such as text, images, videos, music, or code based on patterns it has learned from existing data. Instead of only analyzing data, it generates new outputs that resemble human-created content.
It is usually built using advanced models like Generative Adversarial Networks, Variational Autoencoders, and Transformer Models.

Primary Use Cases of Generative AI Across Industries
1. Healthcare - AI models help researchers simulate new medicines faster.
2. Marketing & Advertising  - Companies use AI to produce large volumes of marketing content quickly.
3. Software Development - Tools like GitHub Copilot help developers write code faster.
4. Media & Entertainment - Generative AI can create visual effects and digital characters.
5. Finance & Banking - Banks use AI to generate reports and analyze financial data quickly.
6. Education - AI helps create customized learning experiences for students.
And many more Industries.

2. Explain the role of probabilistic modeling in generative models. How do these models differ from discriminative models?

Probabilistic modeling is the foundation of generative models. These models try to learn the probability distribution of the data so they can generate new samples similar to the original data.
P(X,Y) where X = input data and Y = labels or outputs
By learning this distribution, the model can:
Generate new data samples similar to the training data.Model uncertainty in predictions.Understand how the data is generated.

Generative Model - IT is  Probability learned Learn P(X, Y) (joint distribution). It Goals is Model how data is generated. it Capable of to generate new data samples. ex- GAN, VAE, Naive Bayes
Discriminative Model - It Learn P(Y | X) (conditional probability). It is use to Directly classify or predict labels. it Cannot generate data. ex- Logistic Regression, SVM, Neural Networks

3. What is the difference between Autoencoders and Variational
Autoencoders (VAEs) in the context of text generation?

An Autoencoder is a neural network used to compress input data into a smaller representation (latent vector) and then reconstruct the original data.
Structure of an Autoencoder
Encoder , Latent Space , Decoder
In text generation, autoencoders can: Learn representations of sentences or documents.Reconstruct input text from encoded vectors.
Help in tasks like text summarization, denoising, and feature extraction.
Variational Autoencoders (VAE) -  A Variational Autoencoder (VAE) is an extension of the autoencoder that uses probabilistic modeling to generate new data.Instead of mapping input to a fixed latent vector, a VAE learns a probability distribution of the latent space (usually Gaussian).
In text generation, VAEs can:Generate new sentences similar to training data.Produce diverse text outputs. Model uncertainty and variability in language

4. Describe the working of attention mechanisms in Neural Machine
Translation (NMT). Why are they critical?

In Neural Machine Translation (NMT), the attention mechanism allows the model to focus on the most relevant words in the input sentence while generating each word in the translated sentence.

Traditional NMT systems used an encoder–decoder architecture where the encoder compresses the entire input sentence into a single vector. This often caused information loss, especially for long sentences.

The attention mechanism solves this problem by allowing the decoder to look at different parts of the input sentence dynamically during translation.
Attention Works in NMT
1. Encoding the Input Sentence
2. Calculating Attention Scores
3. Creating Attention Weights
4. Context Vector Creation
5. Generating the Output Word

5. What ethical considerations must be addressed when using generative AI for creative content such as poetry or storytelling?

When using Generative AI for creative content like poetry or storytelling, several ethical considerations must be addressed to ensure responsible and fair use.
1. Copyright and Intellectual Property
Generative AI models are often trained on large datasets that may include copyrighted works. This raises questions about:
-Who owns the generated content?
-Whether AI outputs resemble existing works too closely.
-Fair compensation for original creators.
2. Plagiarism and Originality
AI-generated creative content should not replicate existing works without attribution. Ethical use requires:
-Ensuring originality of generated content
-Avoiding direct copying of existing literary works
-Transparency about AI assistance
3. Bias and Fair Representation
Generative AI systems learn from historical data, which may contain social or cultural biases. As a result:
-Stories may reinforce stereotypes.
-Certain groups may be unfairly represented.
Developers must monitor and reduce these biases to ensure fair and inclusive storytelling.
4. Transparency and Disclosure
Readers should know whether content was written by humans or generated by AI. Ethical practice includes:
-Disclosing the use of AI tools in content creation
-Avoiding misleading audiences about authorship

When using generative AI for creative content, key ethical considerations include copyright protection, originality, bias reduction, transparency, prevention of misuse, and respect for human creators. Responsible use ensures that AI enhances creativity without harming creators or audiences.


In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import tensorflow as tf
from tensorflow.keras.layers import Dense, Lambda
from tensorflow.keras.models import Model as KerasModel # Renamed Model to avoid conflict
from tensorflow.keras.losses import mse

# Dataset
sentences = [
"The sky is blue",
"The sun is bright",
"The grass is green",
"The night is dark",
"The stars are shining"
]

# Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts(sentences)
sequences = tokenizer.texts_to_sequences(sentences)
max_len = max(len(seq) for seq in sequences)
padded_sequences = pad_sequences(sequences, maxlen=max_len, padding='post')
vocab_size = len(tokenizer.word_index) + 1

print("Word Index:", tokenizer.word_index)
print("Padded Sequences:\n", padded_sequences)

input_dim = max_len
latent_dim = 2

# Define the VAE as a Keras Model subclass
class VAE(tf.keras.Model):
    def __init__(self, original_dim, latent_dim=2, intermediate_dim=16, **kwargs):
        super().__init__(**kwargs)
        self.encoder_h = Dense(intermediate_dim, activation='relu')
        self.z_mean_layer = Dense(latent_dim)
        self.z_log_var_layer = Dense(latent_dim)

        # Decoder layers
        self.decoder_h = Dense(intermediate_dim, activation='relu')
        self.decoder_output = Dense(original_dim, activation='linear')

        self.latent_dim = latent_dim # Store latent_dim for sampling

    def call(self, inputs):
        # Encoder
        h = self.encoder_h(inputs)
        z_mean = self.z_mean_layer(h)
        z_log_var = self.z_log_var_layer(h)

        # Sampling from latent space
        epsilon = tf.random.normal(shape=(tf.shape(z_mean)[0], self.latent_dim))
        z = z_mean + tf.keras.ops.exp(0.5 * z_log_var) * epsilon

        # Decoder
        h_decoded = self.decoder_h(z)
        outputs = self.decoder_output(h_decoded)

        # Calculate VAE loss components
        reconstruction_loss = mse(inputs, outputs) # This is a KerasTensor of shape (batch_size,)
        kl_loss = -0.5 * tf.keras.ops.mean(
            1 + z_log_var - tf.keras.ops.square(z_mean) - tf.keras.ops.exp(z_log_var), axis=-1
        ) # This is a KerasTensor of shape (batch_size,)

        # Add the combined loss to the model. Keras will aggregate these per-sample losses.
        self.add_loss(reconstruction_loss + kl_loss)

        return outputs # The model's primary output is the reconstruction

# Instantiate and compile the VAE model
vae = VAE(original_dim=input_dim, latent_dim=latent_dim)
vae.compile(optimizer='adam') # No loss specified here, as it's handled via self.add_loss in call

# Build the model explicitly before calling summary()
vae.build(input_shape=(None, input_dim))
vae.summary()

vae.fit(padded_sequences,
        padded_sequences, # target is the input itself for autoencoder
        epochs=200,
        batch_size=2)

reconstructed = vae.predict(padded_sequences)

print("Reconstructed vectors:\n", reconstructed)

Word Index: {'the': 1, 'is': 2, 'sky': 3, 'blue': 4, 'sun': 5, 'bright': 6, 'grass': 7, 'green': 8, 'night': 9, 'dark': 10, 'stars': 11, 'are': 12, 'shining': 13}
Padded Sequences:
 [[ 1  3  2  4]
 [ 1  5  2  6]
 [ 1  7  2  8]
 [ 1  9  2 10]
 [ 1 11 12 13]]


/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'vae_3', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Model: "vae_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_40 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_41 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_42 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_43 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_44 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 78.6313
Epoch 2/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 96.3777  
Epoch 3/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 91.4600  
Epoch 4/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 83.9120 
Epoch 5/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 85.7974 
Epoch 6/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 85.5145 
Epoch 7/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 84.5902 
Epoch 8/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 64.7971 
Epoch 9/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 64.2527 
Epoch 10/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 83.1329  
Epoch 11/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 64.4085 
Epoch 12/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 80.3592  
Epoch 13/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 74.7096 
Epoch 14/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 57.3980 
Epoch 15/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 78.9585 


In [ ]:
from transformers import pipeline

# Load text generation pipeline
translator = pipeline("text-generation", model="gpt2")

text = "Translate the following English text to French and German: \
Artificial Intelligence is transforming many industries."

result = translator(text, max_length=100)

print(result[0]['generated_text'])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Translate the following English text to French and German: Artificial Intelligence is transforming many industries. In 2012, the world's leading private equity firm, Andreessen Horowitz, put together a team of researchers from academia, industry, and government to explore the possibilities of artificial intelligence.

"In the next few years, AI will transform many industries. It's changing the way we work, how we build and maintain the tech, how we sell products and services, how we spend our time, how we interact with others, how we communicate and interact with each other. And this will be the beginning of a new era," said Andreessen Horowitz founder and CEO Vikram Pandit, who heads the Andreessen Horowitz AI group.

The goal of the team is to develop a new way of driving AI through a collaborative process. The team consists of at least seven researchers from institutions based in the United States, Canada, China, and the European Union, and is expected to announce the results of the

In [ ]:
eng_sentences = [
"I am happy",
"She is reading",
"He likes music",
"They play football",
"We love learning"
]

spa_sentences = [
"yo estoy feliz",
"ella esta leyendo",
"el gusta musica",
"ellos juegan futbol",
"nosotros amamos aprender"
]
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Layer

# Define a custom Attention Layer
class BahdanauAttention(Layer):
    def __init__(self, units):
        super(BahdanauAttention, self).__init__()
        self.W1 = tf.keras.layers.Dense(units)
        self.W2 = tf.keras.layers.Dense(units)
        self.V = tf.keras.layers.Dense(1)

    def call(self, query, values):
        # query shape == (batch_size, hidden_size)
        # values shape == (batch_size, max_len, hidden_size)

        # hidden_with_time_axis shape == (batch_size, 1, hidden_size)
        # We are doing this to broadcast addition along the time axis to the attention score
        hidden_with_time_axis = tf.keras.ops.expand_dims(query, 1)

        # score shape == (batch_size, max_len, 1)
        # We get 1 at the last axis because we are applying score to self.V
        # The shape of the tensor before applying self.V is (batch_size, max_len, units)
        score = self.V(tf.keras.activations.tanh(self.W1(values) + self.W2(hidden_with_time_axis)))

        # attention_weights shape == (batch_size, max_len, 1)
        attention_weights = tf.keras.activations.softmax(score, axis=1)

        # context_vector shape == (batch_size, hidden_size)
        context_vector = attention_weights * values
        context_vector = tf.keras.ops.sum(context_vector, axis=1)

        return context_vector, attention_weights

# Tokenize English
eng_tokenizer = Tokenizer(filters='')
eng_tokenizer.fit_on_texts(eng_sentences)
eng_seq = eng_tokenizer.texts_to_sequences(eng_sentences)

# Tokenize Spanish
spa_tokenizer = Tokenizer(filters='')
spa_tokenizer.fit_on_texts(spa_sentences)
spa_seq = spa_tokenizer.texts_to_sequences(spa_sentences)

# Padding
max_len_eng = max(len(s) for s in eng_seq)
max_len_spa = max(len(s) for s in spa_seq)

eng_seq = pad_sequences(eng_seq, maxlen=max_len_eng, padding='post')
spa_seq = pad_sequences(spa_seq, maxlen=max_len_spa, padding='post')

eng_vocab = len(eng_tokenizer.word_index) + 1
spa_vocab = len(spa_tokenizer.word_index) + 1
embedding_dim = 64
units = 128

encoder_inputs = tf.keras.Input(shape=(max_len_eng,))
encoder_embedding = tf.keras.layers.Embedding(eng_vocab, embedding_dim)(encoder_inputs)

encoder_outputs, state_h, state_c = tf.keras.layers.LSTM(
    units, return_sequences=True, return_state=True
)(encoder_embedding)
decoder_inputs = tf.keras.Input(shape=(max_len_spa - 1,))
decoder_embedding = tf.keras.layers.Embedding(spa_vocab, embedding_dim)(decoder_inputs)

decoder_lstm = tf.keras.layers.LSTM(
    units, return_sequences=True, return_state=True
)

# For attention, we need the decoder's hidden state (query) and encoder's outputs (values)
# The `decoder_outputs` from the LSTM will be used as the query to the attention mechanism.
# However, the BahdanauAttention layer typically takes the hidden state of the decoder
# at each time step. Since the decoder is return_sequences=True, decoder_outputs already
# represents the hidden states for each time step. We also need the last hidden state for the initial query.

decoder_outputs, decoder_state_h, decoder_state_c = decoder_lstm(
    decoder_embedding, initial_state=[state_h, state_c]
)

# Instantiate the attention layer
attention_layer = BahdanauAttention(units)

# Apply attention. The query is the decoder's hidden state, values are encoder's outputs.
# For NMT, we usually apply attention with the decoder's previous hidden state
# and all encoder outputs.
# Here, we'll simplify and use the last encoder state and all decoder outputs
# for a conceptual context vector to concatenate.

# A more typical NMT implementation would compute context for each decoder output step.
# For simplicity and to get past the 'attention not defined' error, let's use the last encoder state
# as the query for attention, and encoder_outputs as values for generating a single context vector
# that can be broadcast.
# Let's adjust to be closer to a common use case for attention in seq2seq:
# use the last encoder state (state_h) as initial query and encoder_outputs as values.
# However, in this functional API setup, we want attention to interact with decoder_outputs.
# Let's create a context vector that combines information from encoder_outputs
# with the current decoder_outputs.

# The attention layer expects a query (decoder state) and values (encoder outputs).
# Let's use decoder_outputs (representing states at each decoder timestep) as query for each step.
# This means we need to apply attention for each timestep of the decoder.
# For a simple fix, let's assume `state_h` (last encoder state) acts as a fixed query
# that generates a context vector from `encoder_outputs` once, then broadcast it.
# This is not a full attention mechanism for NMT, but addresses the NameError and allows compilation.
# A more complete NMT attention would integrate this per decoder step.

# Simplified attention application: Use encoder's last hidden state as query
# and all encoder outputs as values to get a global context from the source sentence.
# This context will then be combined with each decoder output.
# This is a simplification; a full attention model would calculate context for each decoder step.
# Let's use `state_h` (encoder's final hidden state) as the query.
context_vector, _ = attention_layer(state_h, encoder_outputs)

# Expand context_vector to match decoder_outputs timestep dimension
context_vector_expanded = tf.keras.ops.expand_dims(context_vector, 1)
context_vector_repeated = tf.keras.ops.tile(context_vector_expanded, [1, tf.keras.ops.shape(decoder_outputs)[1], 1])

concat = tf.keras.layers.Concatenate()([decoder_outputs, context_vector_repeated])

outputs = tf.keras.layers.Dense(spa_vocab, activation="softmax")(concat)
model = tf.keras.Model([encoder_inputs, decoder_inputs], outputs)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()
import numpy as np

decoder_input_data = spa_seq[:, :-1]
decoder_target_data = spa_seq[:, 1:]

model.fit(
    [eng_seq, decoder_input_data],
    decoder_target_data,
    epochs=200,
    batch_size=2
)

Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_15      │ (None, 3)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_10        │ (None, 3, 64)     │      1,024 │ input_layer_15[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_10 (LSTM)      │ [(None, 3, 128),  │     98,816 │ embedding_10[0][… │
│                     │ (None, 128),      │            │                   │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_16      │ (None, 2)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bahdanau_attention… │ [(None, 128),     │     33,153 │ lstm_10[0][1],    │
│ (BahdanauAttention) │ (None, 3, 1)]     │            │ lstm_10[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_11        │ (None, 2, 64)     │      1,024 │ input_layer_16[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expand_dims_3       │ (None, 1, 128)    │          0 │ bahdanau_attenti… │
│ (ExpandDims)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_11 (LSTM)      │ [(None, 2, 128),  │     98,816 │ embedding_11[0][… │
│                     │ (None, 128),      │            │ lstm_10[0][1],    │
│                     │ (None, 128)]      │            │ lstm_10[0][2]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tile_2 (Tile)       │ (None, 2, 128)    │          0 │ expand_dims_3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_2       │ (None, 2, 256)    │          0 │ lstm_11[0][0],    │
│ (Concatenate)       │                   │            │ tile_2[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_62 (Dense)    │ (None, 2, 16)     │      4,112 │ concatenate_2[0]… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 236,945 (925.57 KB)

 Trainable params: 236,945 (925.57 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 25ms/step - accuracy: 0.0000e+00 - loss: 2.7734
Epoch 2/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.4000 - loss: 2.7542
Epoch 3/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.7000 - loss: 2.7369
Epoch 4/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.7000 - loss: 2.7192 
Epoch 5/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.8000 - loss: 2.6993
Epoch 6/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.8000 - loss: 2.6771
Epoch 7/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.8000 - loss: 2.6519
Epoch 8/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.8000 - loss: 2.6210
Epoch 9/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.8000 - loss: 2.5832
Epoch 10/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.8000 - loss: 2.5395
Epoch 11/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.8000 - loss: 2.4825
Epoch 12/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.8000

In [ ]:
poems = [
"Roses are red, violets are blue,",
"Sugar is sweet, and so are you.",
"The moon glows bright in silent skies,",
"A bird sings where the soft wind sighs."
]
!pip install transformers torch

from transformers import pipeline, set_seed

# Load GPT-2 text generation model
generator = pipeline("text-generation", model="gpt2")

# Poetry dataset used as style reference
dataset = [
"Roses are red, violets are blue,",
"Sugar is sweet, and so are you.",
"The moon glows bright in silent skies,",
"A bird sings where the soft wind sighs."
]

# Create prompt using dataset
prompt = "\n".join(dataset) + "\nWrite a similar short poem:\n"

print("PROMPT:\n")
print(prompt)

# Generate poem
set_seed(42)

result = generator(
    prompt,
    max_length=60,
    num_return_sequences=1,
    temperature=0.8
)

generated_text = result[0]['generated_text']

print("\nGenerated Poem:\n")
print(generated_text)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Passing `generation_config` together with generation-related arguments=({'temperature', 'num_return_sequences', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=60) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT:

Roses are red, violets are blue,
Sugar is sweet, and so are you.
The moon glows bright in silent skies,
A bird sings where the soft wind sighs.
Write a similar short poem:


Generated Poem:

Roses are red, violets are blue,
Sugar is sweet, and so are you.
The moon glows bright in silent skies,
A bird sings where the soft wind sighs.
Write a similar short poem:
"Climb the mountains in your heart,
Come to the bottom of the sky,
And sing for us.
To the sun, to the moon, and to the stars."
As you write this poem,
When the sun and the moon have come to the bottom of the sky,
They will sing for you.
And when the sun and the moon have come to the bottom of the sky,
They will sing for you.  But no one sings for them,
If you love them that sing,
And you love them that sing,
You must love all of them.  You must love all of them.  Why does that matter?  Because they do not love you.  They love you because they want you to love them.  Or, to put it another way, they love you because they 

In [ ]:
!pip install transformers torch

from transformers import pipeline, set_seed

# Load pre-trained GPT model
generator = pipeline("text-generation", model="gpt2")

set_seed(42)

# Prompt for story plot and character
prompt = """
You are a creative writing assistant.

Generate:
1. A short story plot
2. A main character description

Genre: Fantasy
"""

# Generate output
result = generator(
    prompt,
    max_length=150,
    num_return_sequences=1,
    temperature=0.9
)

print(result[0]["generated_text"])


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are a creative writing assistant.

Generate:
1. A short story plot
2. A main character description

Genre: Fantasy

Cultural: American

Author: David Dorn (also known as Dorn), best known as The White Elephant.

Why did you become inspired to create a writing exercise and how long did you take to write a single article for this exercise?

David Dorn: I think I have been in the writing program for about a year now. All I have is a short list of topics that I have wanted to write about for a long time. All I had to do was create two pages of a short story before I finished writing it. I am always curious as to what I could possibly do differently, but I never thought of it as writing a single story or a single book. It just seemed like a natural step. I was just excited to write my first short story. Then, when I finished, I got to write this next short story that I had written a year earlier. I then started to do some research about writing, and got to look at the books my professo